# JPEG AI as a Threat to Deepfake Detection

Main notebook. Dataset: **ProGAN** (CNNDetection test set). Detector: **CNNDetection** (pre-trained ResNet50). Codec: **JPEG AI reference software**.

Structure (guidelines): Imports - Globals - Utils - Data - Network - Train - Evaluation.

# Imports

In [ ]:
# (Imports) Standard libraries, torch/torchvision, sklearn metrics, and Google Drive mount
import os, sys, random, subprocess
import numpy as np
import torch
import torchvision.transforms as T
from torchvision.models import resnet50
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_fscore_support, roc_curve)

from google.colab import drive
drive.mount('/content/drive')
print('Imports OK | device:', 'cuda' if torch.cuda.is_available() else 'cpu')

# Globals

Variables used throughout the notebook (paths, bitrates, seed).

In [ ]:
# (Globals) Paths, target bitrates and seed used throughout the notebook
device        = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED          = 42
BPP_LIST      = [0.12, 0.25, 0.50, 0.75]      # target bitrates (bits per pixel)
VALID_IMG_EXT = ('.png', '.jpg', '.jpeg', '.bmp')

DATASET_PATH = '/content/drive/MyDrive/ProGAN_Dataset'         # Test Dataset
TRAIN_DATASET = '/content/drive/MyDrive/ProGAN_Train_Dataset'  # Train Dataset (Fine-Tuning)
PROGAN_ROOT  = '/content/progan_data/progan'                   # ProGAN source (after download)
JPEGAI_DIR   = '/content/jpeg-ai-reference-software'           # JPEG AI reference software
WEIGHTS_DIR  = '/content/drive/MyDrive/cnndetection_weights'   # Baseline Detectors


print('Globals OK | DATASET_PATH =', DATASET_PATH)

# Utils

Support functions (standard detection metrics).

In [ ]:
# Compute metrics for binary detection
def compute_metrics(labels, scores, thr=0.5):
    """Standard metrics for binary detection. scores = P(fake), label fake=1."""
    labels, scores = np.asarray(labels), np.asarray(scores)
    pred = (scores > thr).astype(int)
    real, fake = (labels == 0), (labels == 1)
    pr, rc, f1, _ = precision_recall_fscore_support(labels, pred, average='binary', zero_division=0)
    roc_fpr, roc_tpr, _ = roc_curve(labels, scores)
    eer = float(roc_fpr[np.nanargmin(np.abs(roc_fpr - (1 - roc_tpr)))])
    return dict(
        AUC=roc_auc_score(labels, scores),
        AP=average_precision_score(labels, scores),
        Acc=(pred == labels).mean(),
        BalAcc=0.5 * ((pred[fake] == 1).mean() + (pred[real] == 0).mean()),
        FPR=float((pred[real] == 1).mean()),   # FALSE POSITIVE RATE
        FNR=float((pred[fake] == 0).mean()),   # FALSE NEGATIVE RATE
        Prec=pr, Rec=rc, F1=f1, EER=eer,
    )

print('compute_metrics OK')

In [ ]:
# Functions for the frequency-domain forensic analysis
import os, numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.fftpack import dct

SIZE = 256

# Load a grayscale image and resize to SIZE x SIZE
def load_gray(path, size=SIZE):
    img = Image.open(path).convert('L')
    if img.size != (size, size):
        img = img.resize((size, size))
    return np.asarray(img, float)

# Compute the residual of an image by subtracting a Gaussian-blurred version of itself
def residual(g, sigma=1.0):
    return g - gaussian_filter(g, sigma)          # high-pass: isolates the fine-grained fingerprint

# Compute the average 2D FFT spectrum of the residuals of images in a folder
def avg_fft_residual(folder, n=150):
    acc, k = np.zeros((SIZE, SIZE)), 0
    for f in sorted(os.listdir(folder))[:n]:
        F = np.fft.fftshift(np.abs(np.fft.fft2(residual(load_gray(os.path.join(folder, f))))))
        acc += F; k += 1
    return np.log1p(acc / max(k, 1))              # average 2D spectrum (log)

# Compute the radial power spectral density (PSD) of a 2D Fourier spectrum
def radial_psd(spec2d):
    h, w = spec2d.shape
    y, x = np.indices((h, w))
    r = np.hypot(x - w // 2, y - h // 2).astype(int)
    return np.bincount(r.ravel(), spec2d.ravel()) / np.maximum(np.bincount(r.ravel()), 1)

print('Forensic helpers OK')

In [ ]:
# Scoring helper: run a predict function over one condition (real+fake), return labels + scores
@torch.no_grad()
def scores_with(predict_fn, suffix):
    y, s = [], []
    for cls, lab in [('real', 0), ('fake', 1)]:
        d = os.path.join(DATASET_PATH, f'{cls}_{suffix}')
        if not os.path.isdir(d):
            continue
        for fn in sorted(os.listdir(d)):
            s.append(predict_fn(os.path.join(d, fn))); y.append(lab)
    return np.array(y), np.array(s)

In [ ]:
# Unsharp-mask sharpening (Mitigation Strategy)
from PIL import ImageFilter

SHARP_PERCENT = 100  

def sharpen(img):
    return img.filter(ImageFilter.UnsharpMask(radius=2, percent=SHARP_PERCENT, threshold=3))

# Data

Data management: ProGAN download, JPEG AI setup, real/fake split construction and compression at multiple bitrates.

In [ ]:
# Download the CNNDetection ProGAN test set from Google Drive
FILE_ID = '1NZufxaZrjapdsJE_jjP0LGfh1piBy_9i'   # progan_testset.zip file

if not os.path.isdir(PROGAN_ROOT):
    !rm -rf /content/progan_data /content/progan_testset.zip
    !{sys.executable} -m gdown {FILE_ID} -O /content/progan_testset.zip
    !mkdir -p /content/progan_data
    !unzip -q /content/progan_testset.zip -d /content/progan_data
!ls /content/progan_data/progan | head

In [ ]:
# JPEG AI reference software setup
if not os.path.exists('/content/miniconda/bin/conda'):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
    !bash /content/miniconda.sh -b -p /content/miniconda
os.environ['PATH'] = '/content/miniconda/bin:' + os.environ['PATH']
os.environ.pop('PYTHONPATH', None)
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!apt-get -qq install -y git-lfs doxygen graphviz
!git lfs install
%cd /content
if not os.path.isdir(JPEGAI_DIR):
    !git clone https://gitlab.com/wg1/jpeg-ai/jpeg-ai-reference-software.git
%cd /content/jpeg-ai-reference-software
!git lfs fetch
!git lfs checkout
!make setup_env
!conda env list | grep jpeg_ai_vm || echo 'jpeg_ai_vm env NOT created'
print('JPEG AI setup done.')

In [ ]:
# Pick images disjoint from the test set, to build the fine-tuning training set
import os, random
from PIL import Image

EVAL_N = 150    # how many images per class the test set took (ProGAN_Dataset)

# Gather images containing a specific marker
def gather(marker):
    out = []
    for dp, _, fs in os.walk(PROGAN_ROOT):
        for f in fs:
            p = os.path.join(dp, f)
            if f.lower().endswith(('.png', '.jpg', '.jpeg')) and marker in p:
                out.append(p)
    return sorted(out)

# Build a disjoint training set from the gathered images
def build_disjoint_train(count=100, seed=42):
    real, fake = gather('/0_real/'), gather('/1_fake/')
    rng = random.Random(seed); rng.shuffle(real); rng.shuffle(fake)   # SAME shuffle as the test set
    for cls, lst in [('real', real), ('fake', fake)]:
        print(f'{cls}: available {len(lst)}')
        sel = lst[EVAL_N: EVAL_N + count]                              # DISJOINT range (after the EVAL_N used for test)
        od = os.path.join(TRAIN_DATASET, f'{cls}_original'); os.makedirs(od, exist_ok=True)
        for i, p in enumerate(sel):
            Image.open(p).convert('RGB').save(os.path.join(od, f'{cls}_{i:05d}.png'))
        print(f'  train {cls}: {len(sel)}  (indices {EVAL_N}..{EVAL_N+len(sel)})')

In [ ]:
# Functions: real/fake split construction + JPEG AI compression

# Build a dataset split by marker, saving images to a specified path
def build_split_by_marker(root, dataset_path, n_per_class=None, seed=SEED,
                          real_marker='/0_real/', fake_marker='/1_fake/'):
    def gather(marker):
        out = []
        for dp, _, fs in os.walk(root):
            for f in fs:
                if f.lower().endswith(VALID_IMG_EXT):
                    p = os.path.join(dp, f)
                    if marker in p:
                        out.append(p)
        return sorted(out)
    real, fake = gather(real_marker), gather(fake_marker)
    rng = random.Random(seed); rng.shuffle(real); rng.shuffle(fake)
    if n_per_class is not None:
        real, fake = real[:n_per_class], fake[:n_per_class]
    for cls, lst in [('real', real), ('fake', fake)]:
        od = os.path.join(dataset_path, f'{cls}_original'); os.makedirs(od, exist_ok=True)
        n = 0
        for p in lst:
            try:
                Image.open(p).convert('RGB').save(os.path.join(od, f'{cls}_{n:05d}.png')); n += 1
            except Exception as e:
                print('skip', p, e)
        print(f'{cls}_original: {n} (out of {len(lst)})')

# Run a command in the JPEG AI environment
def _jpegai_run(cmd, jpegai_dir, env='jpeg_ai_vm'):
    r = subprocess.run(['conda', 'run', '-n', env] + cmd, cwd=jpegai_dir, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('JPEG AI failed: ' + r.stderr[-1500:])

# Compress an image using JPEG AI, saving the output PNG and a temporary stream file
def compress_image_jpegai(src, out_png, bpp, jpegai_dir, profile='base', stream_dir='/content/jpegai_streams'):
    os.makedirs(stream_dir, exist_ok=True)
    cfg = ['cfg/tools_off.json', f'cfg/profiles/{profile}.json']
    stream = os.path.join(stream_dir, os.path.splitext(os.path.basename(src))[0] + f'_{bpp}.bin')
    try:
        _jpegai_run(['python', '-m', 'src.reco.coders.encoder', src, stream, '-r', out_png,
                     '--set_target_bpp', str(int(round(bpp * 100))), '--cfg'] + cfg, jpegai_dir)
    finally:
        if os.path.exists(stream): os.remove(stream)

# Compress all images in a dataset split using JPEG AI at specified bitrates
def compress_dataset_jpegai(dataset_path, bpp_list, jpegai_dir):
    for cls in ('real', 'fake'):
        sd = os.path.join(dataset_path, f'{cls}_original')
        imgs = [f for f in sorted(os.listdir(sd)) if f.lower().endswith(VALID_IMG_EXT)]
        print('===', cls, len(imgs), 'images ===')
        for bpp in bpp_list:
            od = os.path.join(dataset_path, f'{cls}_bpp{bpp}'); os.makedirs(od, exist_ok=True)
            for fn in imgs:
                out = os.path.join(od, os.path.splitext(fn)[0] + '.png')
                if os.path.exists(out):
                    continue
                try:
                    compress_image_jpegai(os.path.join(sd, fn), out, bpp, jpegai_dir)
                except Exception as e:
                    print('   ', fn, e)
    print('Compression done.')

print('Data functions OK')

In [ ]:
# Compress the test set with JPEG AI
os.environ['PATH'] = '/content/miniconda/bin:' + os.environ['PATH']
build_split_by_marker(PROGAN_ROOT, DATASET_PATH, n_per_class=200)
compress_dataset_jpegai(DATASET_PATH, BPP_LIST, JPEGAI_DIR)

In [ ]:
# Compress the fine-tuning training set with JPEG AI
os.environ['PATH'] = '/content/miniconda/bin:' + os.environ['PATH']
build_disjoint_train(count=100)                       # build the (disjoint) training split
compress_dataset_jpegai(TRAIN_DATASET, BPP_LIST, JPEGAI_DIR)

In [ ]:
# Paired fine-tuning datasets: 50% original / 50% compressed, per bpp range
import os, random
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Preprocessing for training: center crop, random horizontal flip, normalization
train_tf = transforms.Compose([
    transforms.CenterCrop(224),                 # same preprocessing as eval (no resize)
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Paired dataset class for fine-tuning: 50% original / 50% compressed, per bpp range
class PairedBPPDataset(Dataset):
    def __init__(self, root_dir, bpp_pair, prob_compressed=0.5, transform=None):
        self.root_dir, self.bpp_pair = root_dir, bpp_pair
        self.prob_compressed, self.transform = prob_compressed, transform
        self.samples = []
        for cls, lab in [('real', 0), ('fake', 1)]:
            for fn in os.listdir(os.path.join(root_dir, f'{cls}_original')):
                self.samples.append((fn, cls, lab))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        fn, cls, label = self.samples[idx]
        if random.random() < self.prob_compressed:
            b = random.choice(self.bpp_pair)
            path = os.path.join(self.root_dir, f'{cls}_bpp{b}', fn)
        else:
            path = os.path.join(self.root_dir, f'{cls}_original', fn)
        return self.transform(Image.open(path).convert('RGB')), label

BATCH_SIZE = 16
ds_low  = PairedBPPDataset(TRAIN_DATASET, [0.12, 0.25], 0.5, train_tf)
ds_high = PairedBPPDataset(TRAIN_DATASET, [0.50, 0.75], 0.5, train_tf)
train_loader_low  = DataLoader(ds_low,  batch_size=BATCH_SIZE, shuffle=True)
train_loader_high = DataLoader(ds_high, batch_size=BATCH_SIZE, shuffle=True)
print('low:', len(ds_low), '| high:', len(ds_high))

In [ ]:
# Paired consistency dataset: returns (original, compressed, label) for the same image
import random
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

# Preprocessing for consistency training: center crop, normalization (no random flip)
cons_tf = transforms.Compose([                      # deterministic (no random flip inside)
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Paired dataset class for consistency training: returns (original, compressed, label) for the same image
class PairedConsistencyDataset(Dataset):
    def __init__(self, root_dir, bpp_pair, transform):
        self.root_dir, self.bpp_pair, self.tf = root_dir, bpp_pair, transform
        self.samples = [(fn, cls, lab) for cls, lab in [('real', 0), ('fake', 1)]
                        for fn in os.listdir(os.path.join(root_dir, f'{cls}_original'))]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        fn, cls, label = self.samples[idx]
        orig = Image.open(os.path.join(self.root_dir, f'{cls}_original', fn)).convert('RGB')
        b = random.choice(self.bpp_pair)
        comp = Image.open(os.path.join(self.root_dir, f'{cls}_bpp{b}', fn)).convert('RGB')
        if random.random() < 0.5:                    # SAME flip on both -> keep the pair aligned
            orig = orig.transpose(Image.FLIP_LEFT_RIGHT)
            comp = comp.transpose(Image.FLIP_LEFT_RIGHT)
        return self.tf(orig), self.tf(comp), label

cds_low  = PairedConsistencyDataset(TRAIN_DATASET, [0.12, 0.25], cons_tf)
cds_high = PairedConsistencyDataset(TRAIN_DATASET, [0.50, 0.75], cons_tf)
cloader_low  = DataLoader(cds_low,  batch_size=16, shuffle=True)
cloader_high = DataLoader(cds_high, batch_size=16, shuffle=True)
print('cons low:', len(cds_low), '| cons high:', len(cds_high))

In [ ]:
# Metadata manifest for controlled experimentation
import os, csv

def write_manifest(dataset_path, out_csv=None):
    """
    Scans the dataset directory and generates a CSV manifest containing 
    labels, compression conditions, and absolute paths for all images.
    """

    # Default to saving the manifest directly inside the dataset directory
    out_csv = out_csv or os.path.join(dataset_path, 'manifest.csv')
    rows = []

    # Iterate over both classes mapping them to binary labels
    for cls, lab in [('real', 0), ('fake', 1)]:
        for sub in sorted(os.listdir(dataset_path)):

            # Filter only subdirectories belonging to the current class
            if not (sub.startswith(cls + '_') and os.path.isdir(os.path.join(dataset_path, sub))):
                continue

            # Extract the compression condition (e.g., 'original' or 'bpp0.12')
            cond = sub[len(cls) + 1:]                    # 'original' or 'bpp0.12'

            # Extract the numeric bpp value (leave empty for original images)
            bpp = '' if cond == 'original' else cond.replace('bpp', '')
            d = os.path.join(dataset_path, sub)

            # Append a record for every image found in the current subdirectory
            for fn in sorted(os.listdir(d)):
                rows.append([fn, cls, lab, cond, bpp, os.path.join(d, fn)])

    # Write all collected records to the CSV file
    with open(out_csv, 'w', newline='') as fh:
        w = csv.writer(fh)
        w.writerow(['filename', 'class', 'label', 'condition', 'bpp', 'path'])
        w.writerows(rows)
    print(f'Manifest: {out_csv} ({len(rows)} rows)')

# Generate manifests for both the test set and the fine-tuning set
write_manifest(DATASET_PATH)          # test set
write_manifest(TRAIN_DATASET)         # training set (optional)

# Network

Pre-trained **CNNDetection** detector (Wang et al. 2020): ResNet50 with a single output (logit -> sigmoid = P(fake)). Two variants: `prob0.1` (little data augmentation, more vulnerable to compression) and `prob0.5` (robust). Weights are cached on Drive.

In [ ]:
# Download CNNDetection weights (cached on Drive, only the first time)

os.makedirs(WEIGHTS_DIR, exist_ok=True)
urls = {
    'blur_jpg_prob0.1.pth': 'https://www.dropbox.com/s/h7tkpcgiwuftb6g/blur_jpg_prob0.1.pth?dl=1',
    'blur_jpg_prob0.5.pth': 'https://www.dropbox.com/s/2g2jagq2jn1fd0i/blur_jpg_prob0.5.pth?dl=1',
}

# Download weights only if they are not already cached on Google Drive
for name, url in urls.items():
    dst = os.path.join(WEIGHTS_DIR, name)
    if not os.path.exists(dst):
        !wget -q '{url}' -O '{dst}'
        
!ls -la {WEIGHTS_DIR}

In [ ]:
# Load a detector variant + its prediction function

DETECTOR_VARIANT = 'blur_jpg_prob0.5.pth'   # switch to 'blur_jpg_prob0.1.pth' for the vulnerable variant

cnn = resnet50(num_classes=1)
sd = torch.load(os.path.join(WEIGHTS_DIR, DETECTOR_VARIANT), map_location=device)

# Handle potential nested checkpoint structures
sd = sd.get('model', sd)

# Strip 'module.' prefix from keys if the model was saved with DataParallel
sd = {(k[7:] if k.startswith('module.') else k): v for k, v in sd.items()}

# Load weights and set to evaluation mode
print(cnn.load_state_dict(sd))
cnn = cnn.to(device).eval()

# Define the preprocessing pipeline. 
# Crucially, we use CenterCrop(224) instead of Resize to preserve the native 
# high-frequency pixel structures and artifacts that the detector relies upon.
cnn_tf = T.Compose([
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def cnndet_predict(path):
    x = cnn_tf(Image.open(path).convert('RGB')).unsqueeze(0).to(device)
    return float(torch.sigmoid(cnn(x).squeeze()))   # P(fake)

print('Detector ready:', DETECTOR_VARIANT)

In [ ]:
# Load the base detector into two copies to fine-tune (low/high experts)
from torchvision.models import resnet50

def load_base(variant):
    """Utility function to spawn a fresh instance of the baseline detector."""
    m = resnet50(num_classes=1)
    sd = torch.load(os.path.join(WEIGHTS_DIR, variant), map_location=device)
    sd = sd.get('model', sd)
    sd = {(k[7:] if k.startswith('module.') else k): v for k, v in sd.items()}
    m.load_state_dict(sd)
    return m.to(device)

# Define the base weights to build upon
BASE = 'blur_jpg_prob0.5.pth'          # repeat everything with 'blur_jpg_prob0.1.pth' to train the second pair

# Spawn two independent copies for bitrate-specific fine-tuning
model_low, model_high = load_base(BASE), load_base(BASE)

print('base loaded:', BASE)

In [ ]:
# Load the base detector into two copies for the consistency experts

# Spawn two independent copies for the consistency-regularized training
cmodel_low, cmodel_high = load_base(BASE), load_base(BASE)   # repeat with the other variant

print('consistency experts init from', BASE)

# Train

**Mitigation strategies (Objective 4).** We compare three approaches for making the detector robust to JPEG AI compression:

1. **Fine-tuning** with 50% original / 50% compressed augmentation.
2. **Consistency**: same setup as fine-tuning plus an explicit invariance loss that forces the same output on an image and its compressed version (using the original↔compressed pairs).
3. **Sharpening** — test-time preprocessing, no retraining (defined in Utils, evaluated in the Evaluation section).

Strategies 1 and 2 each produce two bitrate-specialized experts (a *low* expert for bpp 0.12/0.25 and a *high* expert for bpp 0.50/0.75). Run the training once per base detector (`prob0.1` and `prob0.5`).

In [ ]:
# Mitigation strategy 1 (fine-tuning): 50% original / 50% compressed augmentation, low/high experts

import torch.optim as optim
from tqdm import tqdm

EPOCHS, LR = 8, 5e-5

# Binary Cross Entropy with logits for stable P(fake) classification
criterion = torch.nn.BCEWithLogitsLoss()

# Separate optimizers for the low-bitrate and high-bitrate experts
optimizer_low  = optim.Adam(model_low.parameters(),  lr=LR)
optimizer_high = optim.Adam(model_high.parameters(), lr=LR)

def train_expert(model, loader, optimizer, name):
    """
    Fine-tunes a detector expert on a mixture of original and compressed images.
    """
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    loop = tqdm(loader, desc=name, leave=False)

    for images, labels in loop:
        images = images.to(device); labels = labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward(); optimizer.step()

        # Compute metrics for logging
        preds = (outputs > 0.0).float()
        running_loss += loss.item() * images.size(0)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=f'{loss.item():.4f}')

    print(f'  {name:11s} -> loss {running_loss/total:.4f} | acc {correct/total:.4f}')

# Execute the training loop
for epoch in range(EPOCHS):
    print(f'--- Epoch {epoch+1}/{EPOCHS} ---')
    train_expert(model_low,  train_loader_low,  optimizer_low,  'Low expert')
    train_expert(model_high, train_loader_high, optimizer_high, 'High expert')

# Save the fine-tuned weights for later evaluation
tag = BASE.replace('blur_jpg_', '').replace('.pth', '')
torch.save(model_low.state_dict(),  f'/content/drive/MyDrive/expert_low_{tag}.pth')
torch.save(model_high.state_dict(), f'/content/drive/MyDrive/expert_high_{tag}.pth')
print('experts saved for', tag)

In [ ]:
# Mitigation strategy 2 (consistency): BCE(orig) + BCE(comp) + LAMBDA * MSE(sigmoid(orig), sigmoid(comp))

import torch.optim as optim, time
from tqdm import tqdm

# LAMBDA controls the weight of the consistency regularization term
EPOCHS, LR, LAMBDA = 8, 5e-5, 1.0

criterion = torch.nn.BCEWithLogitsLoss()
mse = torch.nn.MSELoss()

opt_low  = optim.Adam(cmodel_low.parameters(),  lr=LR)
opt_high = optim.Adam(cmodel_high.parameters(), lr=LR)

def train_consistency(model, loader, optimizer, name):
    """
    Trains a detector to produce consistent predictions across compression levels.
    The total loss combines standard classification accuracy and output-level invariance.
    """
    model.train(); run_cls = run_cons = 0.0; correct = total = 0
    loop = tqdm(loader, desc=name, leave=False)

    # The dataloader yields paired samples: the original image and its compressed version
    for orig, comp, labels in loop:
        orig, comp = orig.to(device), comp.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()

        # Forward pass for both modalities
        out_o, out_c = model(orig), model(comp)

        # 1. Classification Loss: average BCE on both original and compressed images
        loss_cls  = 0.5 * criterion(out_o, labels) + 0.5 * criterion(out_c, labels)

        # 2. Consistency Loss: enforce invariance at the probability level (MSE on sigmoids)
        loss_cons = mse(torch.sigmoid(out_o), torch.sigmoid(out_c))    # output-level invariance

        # Combined objective
        loss = loss_cls + LAMBDA * loss_cons

        # Backward pass and optimization
        loss.backward(); optimizer.step()

        # Track components for logging
        run_cls += loss_cls.item()*orig.size(0); run_cons += loss_cons.item()*orig.size(0)
        correct += ((out_c > 0).float() == labels).sum().item(); total += labels.size(0)

        loop.set_postfix(cls=f'{loss_cls.item():.3f}', cons=f'{loss_cons.item():.3f}')

    print(f'  {name:11s} -> cls {run_cls/total:.4f} | cons {run_cons/total:.4f} | acc {correct/total:.4f}')

t0 = time.time()
for epoch in range(EPOCHS):
    print(f'--- Epoch {epoch+1}/{EPOCHS} ---')
    train_consistency(cmodel_low,  cloader_low,  opt_low,  'Low (cons)')
    train_consistency(cmodel_high, cloader_high, opt_high, 'High (cons)')
CONS_TRAIN_MIN = (time.time() - t0) / 60
print(f'Consistency training time: {CONS_TRAIN_MIN:.1f} min')

# Save the consistency-trained weights
tag = BASE.replace('blur_jpg_', '').replace('.pth', '')
torch.save(cmodel_low.state_dict(),  f'/content/drive/MyDrive/cons_low_{tag}.pth')
torch.save(cmodel_high.state_dict(), f'/content/drive/MyDrive/cons_high_{tag}.pth')
print('consistency experts saved for', tag)

# Evaluation

Detector evaluation on `original` and at each JPEG AI compression level.

- **Objective 2 — degradation curve:** standard metrics (AUC, AP, Accuracy, Balanced Acc, FPR, FNR, F1, EER) as a function of bitrate, for the base detectors and the mitigated models. To evaluate a fine-tuned model, set `cnn = model_low / model_high / cmodel_*` and re-run the metrics cell.
- **Objective 3 — frequency-domain forensic analysis:** average Fourier spectra and DCT coefficient distributions of noise residuals (real/fake, original vs compressed), to explain the observed drop in accuracy (method follows Cannas et al., arXiv:2412.03261).
- **Objective 4 — mitigation:** sharpening (test-time) vs fine-tuning / consistency.

In [ ]:
# Objective 2 - Standard metrics vs bitrate - base and fine-tuned models
# to evaluate a detector, set DETECTOR_VARIANT with the corresponding detector 

cols = ['AUC', 'AP', 'Acc', 'BalAcc', 'FPR', 'FNR', 'F1', 'EER']
print('CONDITION'.ljust(12) + ' ' + ' '.join(c.rjust(7) for c in cols) + '    n')
results = {}

# Evaluate the model on the uncompressed data and each JPEG AI bitrate
for suf in ['original'] + [f'bpp{b}' for b in BPP_LIST]:

    # Fetch ground truth labels (y) and model predictions (s) for the current condition
    y, s = scores_with(cnndet_predict, suf)
    if len(y) == 0:
        print(suf.ljust(12) + '  (missing)'); continue
    
    # Compute metrics and store them for plotting
    m = compute_metrics(y, s); results[suf] = m

    print(suf.ljust(12) + ' ' + ' '.join(f'{m[c]:7.3f}' for c in cols) + f'    {len(y)}')

In [ ]:
# Objective 4 - Mitigation strategy 3 (sharpening): base vs base+sharpening

@torch.no_grad()
def cnndet_predict_sharp(path):
    """
    Predicts the deepfake probability after applying a sharpening filter.
    This test-time preprocessing aims to restore high-frequency details 
    suppressed by the JPEG AI neural compression.
    """                                              # defined here: depends on the trained detector (cnn, cnn_tf)
    img = sharpen(Image.open(path).convert('RGB'))   # only added step vs cnndet_predict
    
    # Apply standard CNNDetection preprocessing (CenterCrop + Normalize)
    x = cnn_tf(img).unsqueeze(0).to(device)          # same preprocessing as base (CenterCrop 224 + norm)
    return float(torch.sigmoid(cnn(x).squeeze()))

print(f'{"cond":9s} {"AUC base":>9} {"AUC sharp":>10} {"Acc base":>9} {"Acc sharp":>10}')
# Compare baseline performance vs. sharpening-augmented performance
for suf in ['original'] + [f'bpp{b}' for b in BPP_LIST]:
    yb, sb = scores_with(cnndet_predict,       suf)   # base (no preprocessing)
    ys, ss = scores_with(cnndet_predict_sharp, suf)   # base + sharpening
    mb, ms = compute_metrics(yb, sb), compute_metrics(ys, ss)
    print(f'{suf:9s} {mb["AUC"]:9.3f} {ms["AUC"]:10.3f} {mb["Acc"]:9.3f} {ms["Acc"]:10.3f}')

In [ ]:
# Objective 2 - Degradation curve plot

# Plot key metrics as a function of the compression bitrate
plt.figure(figsize=(8, 5))
for mname in ['AUC', 'AP', 'Acc', 'F1']:
    plt.plot(BPP_LIST, [results[f'bpp{b}'][mname] for b in BPP_LIST], 'o-', label=mname)

    # Add a dashed horizontal line representing the baseline performance on 'original' images
    plt.axhline(results['original'][mname], ls='--', alpha=0.25)

# Add a dotted line for random chance (50%)
plt.axhline(0.5, color='k', ls=':', label='chance')

plt.xlabel('bitrate (bpp)'); plt.ylabel('metric'); plt.ylim(0.4, 1.02)
plt.title('Degradation curve - CNNDetection vs JPEG AI')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# Objective 3 - Average 2D Fourier spectrum of residuals: real/fake, original vs compressed

# Select the lowest bitrate (strongest compression) to highlight the maximum degradation effect
BPP = 0.12

conds = [('real', 'original'), ('fake', 'original'), ('real', f'bpp{BPP}'), ('fake', f'bpp{BPP}')]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Compute and plot the 2D power spectrum for each class/condition pair
for ax, (cls, suf) in zip(axes, conds):
    S = avg_fft_residual(os.path.join(DATASET_PATH, f'{cls}_{suf}'))
    ax.imshow(S, cmap='viridis'); ax.set_title(f'{cls} - {suf}'); ax.axis('off')
    
plt.suptitle('Average Fourier spectrum of noise residuals'); plt.tight_layout(); plt.show()

In [ ]:
# Objective 3 - Radial power spectrum of residuals (1D)
BPP = 0.12
spec = {}

# Compute the 1D radial profile of the 2D spectrum to analyze frequency decay
for cls in ['real', 'fake']:
    for suf in ['original', f'bpp{BPP}']:
        spec[(cls, suf)] = radial_psd(avg_fft_residual(os.path.join(DATASET_PATH, f'{cls}_{suf}')))

# Plot the spectral densities: solid lines for original, dashed for compressed
plt.figure(figsize=(8, 5))
for (cls, suf), p in spec.items():
    ls = '-' if suf == 'original' else '--'
    plt.semilogy(p, ls, label=f'{cls} {suf}')

plt.xlabel('radial frequency'); plt.ylabel('avg residual power (log)')
plt.title('Radial power spectrum of residuals'); plt.legend(); plt.show()

In [ ]:
# Objective 3 - DCT coefficient distribution on residuals

def dct_logmag_samples(folder, n=150):
    """Extracts and flattens the log-magnitude of DCT coefficients for a set of images."""
    vals = []
    for f in sorted(os.listdir(folder))[:n]:
        g = residual(load_gray(os.path.join(folder, f)))
        D = dct(dct(g, axis=0, norm='ortho'), axis=1, norm='ortho')
        vals.append(np.log(np.abs(D) + 1e-3).ravel())
    return np.concatenate(vals)

BPP = 0.12
plt.figure(figsize=(8, 5))

# Plot histograms to observe how JPEG AI alters the DCT statistical distributions
for cls in ['real', 'fake']:
    for suf, ls in [('original', '-'), (f'bpp{BPP}', '--')]:
        v = dct_logmag_samples(os.path.join(DATASET_PATH, f'{cls}_{suf}'))
        plt.hist(v, bins=80, density=True, histtype='step', linestyle=ls, label=f'{cls} {suf}')

plt.xlabel('log|DCT coeff| (on residual)'); plt.ylabel('density')
plt.title('DCT coefficient distribution'); plt.legend(); plt.show()

In [ ]:
# Objective 3 - Difference of average spectra
# Computing the difference between spectra amplifies subtle generative artifacts (real vs fake) and compression impacts.

def spec2d(cls, suf):
    return avg_fft_residual(os.path.join(DATASET_PATH, f'{cls}_{suf}'))

BPP = 0.12
so_r, so_f = spec2d('real', 'original'), spec2d('fake', 'original')
sc_r, sc_f = spec2d('real', f'bpp{BPP}'), spec2d('fake', f'bpp{BPP}')

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# Define the differential comparisons
panels = [('fake - real (original)', so_f - so_r),
          (f'fake - real (bpp {BPP})', sc_f - sc_r),
          ('real: original - compressed', so_r - sc_r)]

# Plot the differences using a diverging colormap (seismic)
for a, (t, D) in zip(ax, panels):
    m = np.abs(D).max()
    a.imshow(D, cmap='seismic', vmin=-m, vmax=m); a.set_title(t); a.axis('off')
    
plt.tight_layout(); plt.show()